In [1]:
import pandas as pd
import scipy.stats as stats
import altair as alt
import numpy as np
from sklearn.metrics import precision_recall_curve, auc, roc_auc_score
from scipy.stats import fisher_exact
from statsmodels.stats.proportion import proportions_ztest

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [5]:
sge_set=pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260512_SGESplicingSet.xlsx')
curated_set=pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260512_CuratedSplicingSet.xlsx')
clinvar_set = pd.read_excel('./Data/sge_data_for_qc/spliceai_benchmarking/20260512_ClinVarSplicingSet.xlsx')

In [10]:
import itertools

# Maps SGE simplified_consequence labels → curated/ClinVar VEP SO term labels
CONSEQUENCE_MAP = {
    'Intron':           ['intron_variant'],
    'Splice Region':    ['splice_donor_region_variant', 'splice_polypyrimidine_tract_variant',
                         'splice_donor_5th_base_variant', 'splice_region_variant'],
    'Canonical Splice': ['splice_acceptor_variant', 'splice_donor_variant'],
    'Missense':         ['missense_variant'],
    'Synonymous':       ['synonymous_variant'],
}

SGE_LABEL_COL    = 'auth_reported_func_class'
SGE_POS_LABEL    = 'functionally_abnormal'
SGE_NEG_LABEL    = 'functionally_normal'

CURATED_LABEL_COL = 'splice_consequence'
CURATED_POS_LABEL = 'abnormal'
CURATED_NEG_LABEL = 'normal'

CLINVAR_LABEL_COL = 'ClinicalSignificance'
CLINVAR_POS_LABEL = 'PLP'
CLINVAR_NEG_LABEL = 'BLB'
CLINVAR_CONS_COL  = 'first_consequence'


def build_confusion_matrix(df, label_col, pos_label, neg_label,
                            score_col='maxSpliceAI', threshold=0.2):
    """Returns [[TN, FP], [FN, TP]] at a fixed threshold."""
    sub = df[df[label_col].isin([pos_label, neg_label])].dropna(subset=[score_col])
    pos = sub[sub[label_col] == pos_label]
    neg = sub[sub[label_col] == neg_label]
    tp = int((pos[score_col] >= threshold).sum())
    fn = int((pos[score_col] <  threshold).sum())
    tn = int((neg[score_col] <  threshold).sum())
    fp = int((neg[score_col] >= threshold).sum())
    return np.array([[tn, fp], [fn, tp]])


def compare_components(cms, labels, bonferroni_factor=None):
    """
    Compare sensitivity and specificity across N datasets.
    cms:              list of [[TN, FP], [FN, TP]] confusion matrices
    labels:           list of dataset names, same length as cms
    bonferroni_factor: if None (default), correction = n_eligible_pairs * 2.
                      Pass a fixed value (e.g. 6 for 3 datasets) to keep
                      p-values comparable across rows with different eligibility.

    Returns a dict keyed by metric with per-dataset values, n counts,
    and corrected p-values for every eligible pair as 'p_{A}_vs_{B}'.
    """
    n_pairs = len(cms) * (len(cms) - 1) // 2
    factor  = bonferroni_factor if bonferroni_factor is not None else n_pairs * 2

    metric_counts = {
        'sensitivity': lambda cm: (cm[1, 1], cm[1, 0]),  # TP, FN
        'specificity': lambda cm: (cm[0, 0], cm[0, 1]),  # TN, FP
    }

    results = {}
    for metric, get_counts in metric_counts.items():
        entry = {}
        for label, cm in zip(labels, cms):
            s, f = get_counts(cm)
            entry[label] = s / (s + f)
            entry[f'n_{label}'] = int(s + f)

        for la, lb in itertools.combinations(labels, 2):
            s_a, f_a = get_counts(cms[labels.index(la)])
            s_b, f_b = get_counts(cms[labels.index(lb)])
            _, p = proportions_ztest([s_a, s_b], [s_a + f_a, s_b + f_b])
            entry[f'p_{la}_vs_{lb}'] = min(p * factor, 1.0)

        results[metric] = entry

    return results

In [11]:
RNA_FILTERABLE = {'Splice Region', 'Missense', 'Synonymous'}
ALL_DATASET_LABELS = ['SGE', 'Curated', 'ClinVar']

def get_sge_sub(sge_set, group_name, rna_filtered=False):
    sub = sge_set[sge_set['simplified_consequence'] == group_name]
    if rna_filtered:
        # Restrict to variants with RNA data, then keep only RNA-confirmed LoF as positives
        sub = sub[sub['rna_consequence'].notna()]
        sub = sub[
            (sub[SGE_LABEL_COL] == SGE_NEG_LABEL) |
            ((sub[SGE_LABEL_COL] == SGE_POS_LABEL) & (sub['rna_consequence'] == 'low'))
        ]
    return sub


rows = []
for group_name, curated_consequences in CONSEQUENCE_MAP.items():
    cur_sub = curated_set[curated_set['simplified_consequence'].isin(curated_consequences)]
    clv_sub = clinvar_set[clinvar_set[CLINVAR_CONS_COL].isin(curated_consequences)]

    variants = [(group_name, False)]
    if group_name in RNA_FILTERABLE:
        variants.append((f'{group_name} (Low RNA)', True))

    for label, rna_filtered in variants:
        sge_sub = get_sge_sub(sge_set, group_name, rna_filtered)

        dataset_specs = [
            ('SGE',     sge_sub, SGE_LABEL_COL,     SGE_POS_LABEL,     SGE_NEG_LABEL),
            ('Curated', cur_sub, CURATED_LABEL_COL, CURATED_POS_LABEL, CURATED_NEG_LABEL),
            ('ClinVar', clv_sub, CLINVAR_LABEL_COL, CLINVAR_POS_LABEL, CLINVAR_NEG_LABEL),
        ]

        cms_map = {}
        count_info = []
        for ds_label, sub_df, label_col, pos_label, neg_label in dataset_specs:
            n_pos = sub_df[label_col].eq(pos_label).sum()
            n_neg = sub_df[label_col].eq(neg_label).sum()
            count_info.append(f'{ds_label} +{n_pos}/-{n_neg}')
            if min(n_pos, n_neg) >= 5:
                cms_map[ds_label] = build_confusion_matrix(sub_df, label_col, pos_label, neg_label)
            else:
                cms_map[ds_label] = None

        eligible = [k for k, v in cms_map.items() if v is not None]
        if not eligible:
            print(f'Skipping {label}: insufficient counts for all datasets ({", ".join(count_info)})')
            continue

        insufficient = [k for k, v in cms_map.items() if v is None]
        if insufficient:
            print(f'{label}: {insufficient} excluded (insufficient counts), using {eligible} only')

        result = compare_components(
            [cms_map[k] for k in eligible],
            eligible
        )

        for metric, vals in result.items():
            row = {'consequence': label, 'metric': metric}
            for ds_label in ALL_DATASET_LABELS:
                row[ds_label]        = vals.get(ds_label, np.nan)
                row[f'n_{ds_label}'] = vals.get(f'n_{ds_label}', np.nan)
            for la, lb in itertools.combinations(ALL_DATASET_LABELS, 2):
                row[f'p_{la}_vs_{lb}'] = vals.get(f'p_{la}_vs_{lb}', np.nan)
            rows.append(row)

comparison_df = pd.DataFrame(rows)

fmt = lambda x: f'{x:.3f}'
sensitivity_df = comparison_df[comparison_df['metric'] == 'sensitivity'].drop(columns='metric').reset_index(drop=True)
specificity_df = comparison_df[comparison_df['metric'] == 'specificity'].drop(columns='metric').reset_index(drop=True)

print('=== Sensitivity ===')
print(sensitivity_df.to_string(index=False, float_format=fmt))
print()
print('=== Specificity ===')
print(specificity_df.to_string(index=False, float_format=fmt))

Intron: ['Curated'] excluded (insufficient counts), using ['SGE', 'ClinVar'] only
Canonical Splice: ['Curated'] excluded (insufficient counts), using ['SGE', 'ClinVar'] only
Missense: ['ClinVar'] excluded (insufficient counts), using ['SGE', 'Curated'] only
Missense (Low RNA): ['ClinVar'] excluded (insufficient counts), using ['SGE', 'Curated'] only
=== Sensitivity ===
            consequence   SGE  n_SGE  Curated  n_Curated  ClinVar  n_ClinVar  p_SGE_vs_Curated  p_SGE_vs_ClinVar  p_Curated_vs_ClinVar
                 Intron 0.319    342      NaN        NaN    0.586     29.000               NaN             0.007                   NaN
          Splice Region 0.747    747    0.873     71.000    0.917    337.000             0.106             0.000                 1.000
Splice Region (Low RNA) 0.840     25    0.873     71.000    0.917    337.000             1.000             1.000                 1.000
       Canonical Splice 0.995   1033      NaN        NaN    0.995   2914.000            